# Fine-Tuning AI Models, Managing Datasets, and Saved Prompts

This notebook demonstrates an end-to-end workflow for preparing data and prompts for an AI classification task. It loads datasets from Hugging Face and Kaggle, transforms a spam-message dataset into the JSONL chat format required for supervised fine-tuning, and uses a saved prompt with the OpenAI Responses API to classify messages as **spam** or **ham**.

In [1]:
!pip install -q datasets kagglehub pandas

## Load a dataset (`MongoDB/whatscooking.restaurants`) from HuggingFace

In [2]:
from datasets import load_dataset

hf_dataset = load_dataset("MongoDB/whatscooking.restaurants")

README.md:   0%|          | 0.00/5.78k [00:00<?, ?B/s]

whatscooking.restaurants.json: reconstructing file:   0%|          |  0.00B /  147MB            

whatscooking.restaurants.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25361 [00:00<?, ? examples/s]

In [3]:
# Save the dataset splits as CSV files.
for split_name, split_rows in hf_dataset.items():
    hf_df = split_rows.to_pandas()
    hf_df.to_csv(f"/content/mongodb_restaurants_{split_name}.csv", index=False)

## Load a dataset (`team-ai/spam-text-message-classification`) from Kaggle

In [4]:
import kagglehub

kaggle_df = kagglehub.dataset_load(kagglehub.KaggleDatasetAdapter.PANDAS, "team-ai/spam-text-message-classification", "SPAM text message 20170820 - Data.csv")

100%|Р Р†РІР‚вЂњРІвЂљВ¬Р Р†РІР‚вЂњРІвЂљВ¬Р Р†РІР‚вЂњРІвЂљВ¬Р Р†РІР‚вЂњРІвЂљВ¬Р Р†РІР‚вЂњРІвЂљВ¬Р Р†РІР‚вЂњРІвЂљВ¬Р Р†РІР‚вЂњРІвЂљВ¬Р Р†РІР‚вЂњРІвЂљВ¬Р Р†РІР‚вЂњРІвЂљВ¬Р Р†РІР‚вЂњРІвЂљВ¬| 474k/474k [00:00<00:00, 82.3MB/s]


In [11]:
kaggle_df

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will Р вЂњРЎВ b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


## Convert the dataset to JSONL for fine-tuning

The [chat completions](https://developers.openai.com/api/docs/guides/supervised-fine-tuning#build-your-dataset) format is used.

Example:
```json
{
    "messages": [
        { "role": "user", "content": "What is the capital of France?" },
        { "role": "assistant", "content": "The capital of France is Paris." }
    ]
}
```

In [7]:
SYSTEM_MESSAGE = "Classify the message strictly as 'spam' or 'ham'. Return only one word."
fine_tuning_df = kaggle_df.apply(
    lambda x: [
        { "role": "developer", "content": SYSTEM_MESSAGE },
        { "role": "user", "content": f"Classify the message: \"{x["Message"]}\"" },
        { "role": "assistant", "content": x["Category"] }
    ],
    axis=1
)

fine_tuning_df = fine_tuning_df.to_frame(name="messages")

In [8]:
fine_tuning_df.to_json("/content/fine_tuning.jsonl", index=False, orient="records", lines=True)

# OpennAI platform

In [16]:
from openai import OpenAI
from google.colab import userdata

In [ ]:
api_key = userdata.get('open-ai-key')
client = OpenAI(api_key=api_key)

In [13]:
email_spam_detection_prompt = """You are an email classification assistant. Your task is to classify short email messages into one of two categories:
- ham - legitimate, normal email communication.
- spam - unsolicited, promotional, fraudulent, or malicious email.
Rules:
- Respond with only one word: ham or spam.
- Do not include explanations.
- Base your decision only on the content of the email message.
- If uncertain, choose the most likely category.

Classify the following email:
{{message}}"""

In [15]:


response = client.responses.create(
  prompt={
    "id": "pmpt_6a69b9b1ebc481979811bd210160d39f07aa7b5066b0997d",
    "version": "1",
    "variables": {
      "messages": "example messages"
    }
  }
)

In [18]:
from pprint import pprint

def print_response(response):
  print(f"Response id: {response.id}")
  print(f"Input tokens: {response.usage.input_tokens} ({response.usage.input_tokens_details.cached_tokens} cached); Output tokens: {response.usage.output_tokens} ({response.usage.output_tokens_details.reasoning_tokens} reasoning)")
  pprint(response.output)

  print()
  print(f"{'-' * 20} [Text] {'-' * 20}")
  print(response.output_text)

In [19]:
print_response(response)

Response id: resp_003adc4a16a7d2d6006a69b9cf79788192912db1e0455bb2c6
Input tokens: 104 (0 cached); Output tokens: 79 (72 reasoning)
[ResponseReasoningItem(id='rs_003adc4a16a7d2d6006a69b9cfc33c81928de6c25ef37cc9d2', summary=[Summary(text='**Classifying email content**\n\nI\'m thinking about how to classify the email content labeled "example messages." It really seems like just a placeholder, and I\'m trying to decide if it\'s ham or spam based only on this content. It\'s so short and generic. Maybe it leans more toward ham since it doesn\'t appear promotional or malicious. I have to output just one word, so I\'ll go with "ham."', type='summary_text')], type='reasoning', content=[], encrypted_content='gAAAAABqabnSKeY3dniN9eQQJhEvX9PXtjP3-Poz3ZAxjrVmzykcWEvhtQ27iZEyN7UakanXUmmbRMN_TOK8vu5N82--L08f4yxtiTSpvw_pjKq1UcEsyV2Qfn60RMxRox7kYMEOhiEvCy8Hqu1QcMVfiWHr0p87CCw3FLTKm0S2k0cKSVw_QrxOGWGvo1CT346kLSjKqrrPgOGmehnQGfnOOiB7GUvW-R6ZAdF6c8TqIOY0yyCBCLNV4cBDLlcdW79ZOxNJiAFFlVyTgC7AK71s7w1eER19v9F

# Create an OpenAI evaluation dataset and graders

OpenAI eval datasets use a different JSONL shape from supervised fine-tuning files. Each line must expose the values used by the eval under an `item` key. For this classification task, every row contains the email `message` and its expected `label` (`spam` or `ham`).

Example JSONL row:

```json
{"item":{"message":"Congratulations! You won a prize.","label":"spam"}}
```

In [ ]:
# Convert the Kaggle dataframe to the JSONL structure expected by OpenAI Evals.
EVAL_DATASET_PATH = "/content/spam_eval_dataset.jsonl"

eval_dataset_df = pd.DataFrame({
    "item": kaggle_df.apply(
        lambda row: {
            "message": str(row["Message"]),
            "label": str(row["Category"]).strip().lower(),
        },
        axis=1,
    )
})

eval_dataset_df.to_json(
    EVAL_DATASET_PATH,
    orient="records",
    lines=True,
    force_ascii=False,
)

eval_dataset_df.head()

## Upload the JSONL file as an evaluation dataset

Upload the file through the Files API with `purpose="evals"`. The returned file ID is used as the source of an evaluation run.

In [ ]:
with open(EVAL_DATASET_PATH, "rb") as dataset_file:
    eval_dataset_file = client.files.create(
        file=dataset_file,
        purpose="evals",
    )

print(f"Evaluation dataset file ID: {eval_dataset_file.id}")
print(f"Filename: {eval_dataset_file.filename}")
print(f"Status: {eval_dataset_file.status}")

## Create graders and the evaluation

Graders are added as `testing_criteria` when the evaluation is created. Template variables under `item` come from each JSONL row, while `sample.output_text` contains the model response.

The first grader requires an exact match. The second is a Python grader that ignores surrounding whitespace and letter case, which helps distinguish a correct classification from a formatting-only error.

In [ ]:
strict_match_grader = {
    "type": "string_check",
    "name": "Strict spam/ham match",
    "input": "{{sample.output_text}}",
    "reference": "{{item.label}}",
    "operation": "eq",
}

normalized_match_grader = {
    "type": "python",
    "name": "Normalized spam/ham match",
    "pass_threshold": 1.0,
    "source": """
def grade(sample: dict, item: dict) -> float:
    predicted = str(sample.get("output_text", "")).strip().lower()
    expected = str(item.get("label", "")).strip().lower()
    return 1.0 if predicted == expected else 0.0
""",
}

spam_eval = client.evals.create(
    name="Spam message classification",
    data_source_config={
        "type": "custom",
        "item_schema": {
            "type": "object",
            "properties": {
                "message": {"type": "string"},
                "label": {"type": "string", "enum": ["spam", "ham"]},
            },
            "required": ["message", "label"],
            "additionalProperties": False,
        },
        "include_sample_schema": True,
    },
    testing_criteria=[strict_match_grader, normalized_match_grader],
    metadata={"task": "spam-classification"},
)

print(f"Eval ID: {spam_eval.id}")

## Get the fine-tuned model ID

When a fine-tuning job finishes, OpenAI writes the model name into the job's `fine_tuned_model` field. That value is the model ID you pass to the eval run as `EVAL_MODEL`.

Use this after the fine-tuning job status is `succeeded`. If the job is still running, `fine_tuned_model` will be empty.

In [ ]:
# Option 1: if you already have the fine-tuning job object from a previous cell.
# fine_tuning_job = client.fine_tuning.jobs.create(...)

try:
    fine_tuning_job_id = fine_tuning_job.id
except NameError:
    # Option 2: paste the job ID from the OpenAI dashboard or from the create-job response.
    fine_tuning_job_id = "ftjob_..."

fine_tuning_job_status = client.fine_tuning.jobs.retrieve(fine_tuning_job_id)

print(f"Job status: {fine_tuning_job_status.status}")
print(f"Fine-tuned model ID: {fine_tuning_job_status.fine_tuned_model}")

if fine_tuning_job_status.status == "succeeded":
    EVAL_MODEL = fine_tuning_job_status.fine_tuned_model
    print(f"Use this for evaluation: EVAL_MODEL = {EVAL_MODEL}")
else:
    print("The model ID will be available after the fine-tuning job succeeds.")

## Run the evaluation against a model

This run asks a model to classify every message in the uploaded dataset and then applies both graders. Replace `EVAL_MODEL` with a fine-tuned model ID to evaluate the fine-tuned model instead.

In [ ]:
EVAL_MODEL = "gpt-4.1-mini"  # Or use a fine-tuned model ID: ft:...

eval_run = client.evals.runs.create(
    spam_eval.id,
    name=f"{EVAL_MODEL} spam classification",
    data_source={
        "type": "completions",
        "source": {
            "type": "file_id",
            "id": eval_dataset_file.id,
        },
        "input_messages": {
            "type": "template",
            "template": [
                {
                    "role": "developer",
                    "content": SYSTEM_MESSAGE,
                },
                {
                    "role": "user",
                    "content": 'Classify the message: "{{item.message}}"',
                },
            ],
        },
        "model": EVAL_MODEL,
        "sampling_params": {"temperature": 0},
    },
)

print(f"Run ID: {eval_run.id}")
print(f"Status: {eval_run.status}")
print(f"Report: {eval_run.report_url}")

In [ ]:
# Run this cell again until the status is "completed" or "failed".
run_status = client.evals.runs.retrieve(eval_run.id, eval_id=spam_eval.id)

print(f"Status: {run_status.status}")
print(f"Results: {run_status.result_counts}")
print(f"Report: {run_status.report_url}")

In [ ]:
# Compute the final evaluation score after the run is completed.
run_status = client.evals.runs.retrieve(eval_run.id, eval_id=spam_eval.id)

if run_status.status != "completed":
    print(f"Run is not finished yet. Current status: {run_status.status}")
else:
    counts = run_status.result_counts
    total = counts.total
    passed = counts.passed
    failed = counts.failed
    errored = counts.errored

    final_score = passed / total if total else 0
    print(f"Final score: {final_score:.2%}")
    print(f"Passed: {passed} / {total}")
    print(f"Failed: {failed}")
    print(f"Errored: {errored}")

    if run_status.per_testing_criteria_results:
        print("\nScore by grader:")
        for result in run_status.per_testing_criteria_results:
            grader_total = result.passed + result.failed
            grader_score = result.passed / grader_total if grader_total else 0
            print(f"- {result.testing_criteria}: {grader_score:.2%} ({result.passed}/{grader_total})")